In [ ]:
from src.web_scraper.sitemap_scraper import scrape_sitemap, process_sitemap_urls, scrape_multiple_links, save_raw_html_outputs
from src.utils.utils import generate_directories

In [ ]:
# Generate directories needed for project
# TODO: Get rid of this, generate the directories in the function itself
generate_directories()

### Scrape Sitemap & Save Locally
This will be used to embed the raw HTMLs

#### TODO: 
1. Make an orchestration function for this 
2. Save sitemap locally so I don't have to keep rerunning and scraping stuff I already have (for now)
3. Add missing links that I know we somehow missed in the site map

In [ ]:
# sitemap_url = 'https://wildfrostwiki.com/sitemap.xml'
# sitemap_urls = scrape_sitemap(sitemap_url)

In [ ]:
# urls = process_sitemap_urls(sitemap_urls)

In [ ]:
# html_outputs = await scrape_multiple_links(urls)

In [ ]:
# raw_html_subdirectory = 'raw_htmls'
# save_raw_html_outputs(html_outputs, raw_html_subdirectory)

### Scrape and save the sites in accordance to the ontology

Schemas:
1. Cards Schema: https://wildfrostwiki.com/index.php?title=Baby_Snowbo; grab the table at the end, create a folder structure based on that
2. Fights & Boss Battles: https://wildfrostwiki.com/The_Bog_Berries; grab the Map Events Table at the end
3. Charms: https://wildfrostwiki.com/Charms; another table
4. Stats, Buffs, Debuffs: https://wildfrostwiki.com/Stats
5. Keywords: https://wildfrostwiki.com/Keywords; relations between stats

In [ ]:
from src.data_processing.cards import CardType, CardInfo

In [ ]:
from src.data_processing.generate_schemas import generate_card_type_html_schema

In [ ]:
card_type_schema = generate_card_type_html_schema()

In [ ]:
import os
import json

filename = '../data/schemas/'
schema_filename = os.path.join(filename,'card_type_schema.json')
os.makedirs(filename, exist_ok=True)

with open(schema_filename,'w',encoding='utf-8') as f:
    json.dump(card_type_schema, f, indent=4)

In [ ]:
for k, v in card_type_schema.items():
    print(f'{k}: {v}')

In [ ]:
base_url = 'https://wildfrostwiki.com'

In [ ]:
import re

def clean_name_for_url(name: str) -> str:
    """Clean card name for use in URLs by replacing spaces with underscores"""
    return re.sub(r'\s+', '_', name)

In [ ]:
# Need to make sure that the sub_directory is part of the link. Might just use a tuple and use tuple unpacking into the scrape multiple links function
# TODO: 
#   1. Create a dictionary where each card link is placed in a subdirectory as a key 
#   2. Take the dictionary, scrape 

card_infos  = []
for card_type, cards in card_type_schema.items():
    if card_type == 'leaders':
        continue

    for card_name in cards:
        cleaned_name = clean_name_for_url(card_name)
        card_info = CardInfo(
            card_name=card_name,
            card_type=CardType(card_type),
            card_url=f'{base_url}/{cleaned_name}'
        )
        card_infos.append(card_info)

for c in card_infos:
    print(f'{c.card_name} {c.card_url}\n')

In [ ]:
urls = [card.card_url for card in card_infos]

In [ ]:
card_types_html_outputs = await scrape_multiple_links(urls, max_concurrent=50)

In [ ]:
for card_info, html in zip(card_infos, card_types_html_outputs):
    card_info.card_html = html
    if card_info.card_html is not None:
        card_info.save_html()
        card_info.parse_html()

In [ ]:
card_infos

In [ ]:
for c in card_infos:
    print(f'{c}\n')

In [ ]:
for c in card_infos:
    print(f'{c.to_dict()}\n')

### Leaders HTML

In [ ]:
# import requests
# from bs4 import BeautifulSoup

# leaders_url = 'https://wildfrostwiki.com/Leaders'
# response = requests.get(leaders_url)
# response.raise_for_status()

# soup = BeautifulSoup(response.text, 'html.parser')

# print(soup.prettify())

### Tribe Exclusivity Check

#### Tribe Assignment For Companions

In [ ]:
import requests
from bs4 import BeautifulSoup, Comment

leaders_url = 'https://wildfrostwiki.com/Companions'
response = requests.get(leaders_url)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'html.parser')

print(soup.prettify())

# Remove comments from HTML
comments = soup.find_all(string=lambda text: isinstance(text, Comment))
for comment in comments:
    comment.extract()

# Save cleaned HTML to file
with open('../data/companion_tribe_check.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())


In [ ]:
tables = soup.find_all('table', {'class': 'wikitable sortable'})

In [ ]:
second_table = tables[1]

In [ ]:
# Find the header row and get the headers
headers = [th.text.strip() for th in second_table.find('tr').find_all('th')]

In [ ]:
headers

In [ ]:
# Find the indices of the desired columns

tribe_lookup = {}

try:
    card_name_index = headers.index('Card Name')
    tribe_exclusive_index = headers.index('Tribe-exclusive?')
except ValueError as e:
    print(f"One of the required headers was not found: {e}")
else:
    # Iterate over each row (skipping the header row)
    for row in second_table.find_all('tr')[1:]:
        cells = row.find_all(['th', 'td'])
        
        if len(cells) > max(card_name_index, tribe_exclusive_index):
            card_name = cells[card_name_index].get_text(strip=True)
            tribe_name = cells[tribe_exclusive_index].get_text(strip=True)

            # Populate the dictionary directly
            tribe_lookup[card_name] = tribe_name

            print(f"Card Name: {card_name}, Tribe-exclusive?: {tribe_name}")

In [ ]:
tribe_lookup

In [ ]:
from src.data_processing.cards import TribeExclusivity

In [ ]:
for card_info in card_infos:
    tribe_name_str = tribe_lookup.get(card_info.card_name)

    if tribe_name_str:
        try:
            # Dynamically find the correct enum member
            matching_enum = next(t for t in TribeExclusivity if t.value == tribe_name_str)
            
            # Assign the enum member to the card's field
            card_info.tribe_exclusivity = matching_enum
            
        except StopIteration:
            # This handles cases where a tribe string exists but doesn't match an enum member.
            print(f"Warning: No matching TribeExclusivity enum found for '{tribe_name_str}' for card '{card_info.card_name}'")

In [ ]:
for c in card_infos:
    print(f'{c.card_name}: {c.tribe_exclusivity}')

#### Tribe Assignment For Items

In [ ]:
import requests
from bs4 import BeautifulSoup

leaders_url = 'https://wildfrostwiki.com/Items'
response = requests.get(leaders_url)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'html.parser')

print(soup.prettify())

# Remove comments from HTML
comments = soup.find_all(string=lambda text: isinstance(text, Comment))
for comment in comments:
    comment.extract()

# Save cleaned HTML to file
with open('../data/item_tribe_check.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())


In [ ]:
tables = soup.find_all('table', {'class': 'wikitable sortable'})

In [ ]:
only_table = tables[0]
only_table

In [ ]:
# Find the header row and get the headers
headers = [th.text.strip() for th in only_table.find('tr').find_all('th')]

In [ ]:
headers

In [ ]:
# Find the indices of the desired columns
tribe_lookup_items = {}

try:
    card_name_index = headers.index('Card Name')
    tribe_exclusive_index = headers.index('Tribe-exclusive?')
except ValueError as e:
    print(f"One of the required headers was not found: {e}")
else:
    # Iterate over each row (skipping the header row)
    for row in only_table.find_all('tr')[1:]:
        cells = row.find_all(['th', 'td'])
        
        if len(cells) > max(card_name_index, tribe_exclusive_index):
            card_name = cells[card_name_index].get_text(strip=True)
            tribe_name = cells[tribe_exclusive_index].get_text(strip=True)

            # Populate the dictionary directly
            tribe_lookup_items[card_name] = tribe_name

            print(f"Card Name: {card_name}, Tribe-exclusive?: {tribe_name}")

In [ ]:
tribe_lookup_items

In [ ]:
from src.data_processing.cards import TribeExclusivity

In [ ]:
for card_info in card_infos:
    tribe_name_str = tribe_lookup_items.get(card_info.card_name)

    if tribe_name_str:
        try:
            # Dynamically find the correct enum member
            matching_enum = next(t for t in TribeExclusivity if t.value == tribe_name_str)
            
            # Assign the enum member to the card's field
            card_info.tribe_exclusivity = matching_enum
            
        except StopIteration:
            # This handles cases where a tribe string exists but doesn't match an enum member.
            print(f"Warning: No matching TribeExclusivity enum found for '{tribe_name_str}' for card '{card_info.card_name}'")

In [ ]:
for c in card_infos:
    print(f'{c.card_name}: {c.tribe_exclusivity}')

### Test Neo4j

In [ ]:
from src.neo4j_kg.neo4j_utils import create_neo4j_data

In [ ]:
# Put all the dictionary info of card_infos into a list
# I should make this a function really
cards_dict_data = [card.to_dict() for card in card_infos]

In [ ]:
cards_dict_data

In [ ]:
create_neo4j_data(cards_dict_data)

### Chunking HTML

In [ ]:
from src.data_processing.html_splitter import process_html_files

In [ ]:
# Example usage of the function with a list of file paths.
# Note: The file 'data/structured_outputs/items/Azul Battle Axe.html' must exist for this to run.
sample_filepaths = ['data/structured_outputs/items/Azul Battle Axe.html', 'data/structured_outputs/items/Azul Candle.html']

# Process the files and get all the chunks.
all_document_chunks = process_html_files(sample_filepaths)

# Print a summary of all chunks from all files.
if all_document_chunks:
    print("\n--- All Chunks from All Files ---")
    for i, chunk in enumerate(all_document_chunks):
        print(f"Chunk {i+1}: '{chunk.page_content}...'")
        print(f"Metadata: {chunk.metadata}")
        print("-" * 20)
    print(f"\nTotal chunks returned: {len(all_document_chunks)}")
else:
    print("\nNo chunks were created.")

### Turn chunks into smenatic embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
all_chunk_texts = [chunk.page_content for chunk in all_document_chunks]

In [ ]:
embedding = model.encode(all_chunk_texts)

In [ ]:
chunk = 0
for emd in embedding:
    chunk +=1
    print(f'{chunk}: {emd.shape}')

In [ ]:
embedding[0].shape

In [ ]:
import os
from neo4j import GraphDatabase
from typing import List, Dict, Any
import numpy as np
import time

In [ ]:
def ingest_embeddings_into_neo4j(
    uri: str,
    user: str,
    password: str,
    document_chunks: List[Any],
    embeddings: np.ndarray,
    chunk_label: str = "Document",
    text_property: str = "text",
    embedding_property: str = "embedding"
) -> None:
    """
    Ingests document chunks and their embeddings into a Neo4j database using a batching approach.
    """
    driver = GraphDatabase.driver(uri, auth=(user, password))
    driver.verify_connectivity()
    print("Connection to Neo4j successful.")
    
    with driver.session() as session:
        # Use MERGE to avoid creating duplicate nodes
        cypher_query = f"""
        UNWIND $data AS item
        MERGE (d:{chunk_label} {{
            {text_property}: item.text
        }})
        ON CREATE SET d.{embedding_property} = item.embedding
        """
        
        data_to_ingest = [
            {
                "text": chunk.page_content, 
                "embedding": embedding.tolist()
            } 
            for chunk, embedding in zip(document_chunks, embeddings)
        ]
        
        session.run(cypher_query, parameters={"data": data_to_ingest})
    
    driver.close()
    print("Data ingestion complete.")

In [ ]:
def create_vector_index(
    uri: str,
    user: str,
    password: str,
    index_name: str,
    embedding_dimension: int,
    similarity_function: str = "cosine"
) -> None:
    """
    Creates a vector index in Neo4j if one does not already exist.
    """
    driver = GraphDatabase.driver(uri, auth=(user, password))
    with driver.session() as session:
        # Check if the index already exists to avoid errors
        index_exists_query = "SHOW INDEXES YIELD name WHERE name = $name"
        if session.run(index_exists_query, name=index_name).single():
            print(f"Vector index '{index_name}' already exists. Skipping creation.")
            return

        create_query = f"""
        CREATE VECTOR INDEX `{index_name}` IF NOT EXISTS
        FOR (d:Document) ON (d.embedding)
        OPTIONS {{
          indexConfig: {{
            `vector.dimensions`: {embedding_dimension},
            `vector.similarity_function`: "{similarity_function}"
          }}
        }}
        """
        session.run(create_query)
        print(f"Vector index '{index_name}' successfully created.")
    driver.close()

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='configs/.env')

username = os.getenv('TEST_EMBEDDING_NEO4J_USERNAME')
password = os.getenv('TEST_EMBEDDING_NEO4J_PASSWORD')

In [ ]:
URI = "bolt://localhost:7687"
USERNAME = os.getenv('TEST_EMBEDDING_NEO4J_USERNAME')
PASSWORD = os.getenv('TEST_EMBEDDING_NEO4J_PASSWORD')

In [ ]:
EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'
EMBEDDING_DIMENSION = 384

In [ ]:
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Any

def get_retrieved_chunks(
    query: str,
    uri: str,
    user: str,
    password: str,
    embedding_model: SentenceTransformer,
    k: int = 5
) -> List[Dict[str, Any]]:
    """
    Retrieves the top-k most relevant document chunks from Neo4j based on a user query.
    """
    # Step 1: Embed the user's query
    query_embedding = embedding_model.encode(query).tolist()
    
    driver = GraphDatabase.driver(uri, auth=(user, password))
    with driver.session() as session:
        # Step 2: Perform a vector similarity search in Neo4j using the vector index.
        search_query = """
        CALL db.index.vector.queryNodes('document-embeddings', $k, $query_embedding)
        YIELD node, score
        RETURN node.text AS text, score
        ORDER BY score DESC
        """
        
        results = session.run(
            search_query, 
            query_embedding=query_embedding, 
            k=k
        )
        
        # Step 3: Extract the retrieved context and return it.
        retrieved_chunks = [
            {"text": record["text"], "score": record["score"]}
            for record in results
        ]
    
    driver.close()
    return retrieved_chunks


In [ ]:
# 3. Ingest the data into Neo4j.
ingest_embeddings_into_neo4j(
    uri=URI,
    user=USERNAME,
    password=PASSWORD,
    document_chunks=all_document_chunks,
    embeddings=embedding
)

# 4. Create the vector index after ingestion.
create_vector_index(
    uri=URI,
    user=USERNAME,
    password=PASSWORD,
    index_name="document-embeddings",
    embedding_dimension=EMBEDDING_DIMENSION
)

print("Waiting for index to be fully populated...")
time.sleep(5) 

# 5. Perform the vector search (RAG retrieval step).


In [ ]:
query = "What kind of card is the Azul Candle?"
retrieved_chunks = get_retrieved_chunks(
    query=query,
    uri=URI,
    user=USERNAME,
    password=PASSWORD,
    embedding_model=model
)

### Ollama response

In [ ]:
import ollama

def get_rag_response_with_ollama(
    user_query: str,
    retrieved_chunks: List[Dict[str, Any]],
    model_name: str = "mistral"
) -> str:
    """
    Generates a response using Ollama by combining a user query with retrieved chunks.
    
    Args:
        user_query: The user's original question.
        retrieved_chunks: A list of the most relevant text chunks from your Neo4j search.
        model_name: The name of the Ollama model to use (e.g., 'mistral').
    
    Returns:
        The generated response from the Ollama model.
    """
    if not retrieved_chunks:
        return "Sorry, I couldn't find any relevant information to answer your question."
    
    # Concatenate the text from all retrieved chunks into a single context string.
    context = "\n\n".join([chunk['text'] for chunk in retrieved_chunks])
    
    # Construct the messages list for the Ollama chat API.
    messages = [
        {
            'role': 'system',
            'content': 'You are a helpful assistant. Use the following context to answer the question. If the information is not in the context, state that you cannot answer from the provided information.'
        },
        {
            'role': 'user',
            'content': f"Context: {context}\n\nQuestion: {user_query}"
        }
    ]
    
    # Call the Ollama chat API to get a response.
    try:
        response = ollama.chat(
            model=model_name,
            messages=messages
        )
        return response['message']['content']
    except Exception as e:
        return f"Error communicating with Ollama: {e}"



In [ ]:
query

In [ ]:
print("\n--- Retrieved Chunks from Neo4j ---")
if retrieved_chunks:
    for chunk in retrieved_chunks:
        print(f"Score: {chunk['score']:.4f}")
        print(f"Text: {chunk['text']}\n")
else:
    print("No chunks were retrieved. Check your data ingestion and index creation.")

In [ ]:
from pprint import pprint

final_response = get_rag_response_with_ollama(
    user_query=query,
    retrieved_chunks=retrieved_chunks,
    model_name="mistral"
)
print("\n--- Final RAG Response from Ollama ---")
pprint(final_response)

### Test ollama embedding

In [ ]:
import ollama

response = ollama.embed(
    model='embeddinggemma:latest',
    input="My text here"
)

embeddings = response['embeddings']
print(embeddings)